# 📊 Olist — Sales & Delivery Analysis

**Dataset:** Brazilian E-Commerce (Olist, 2016–2018)  
**Tablas:** `orders`, `customers`, `payments`  
**Objetivo:** Análisis de rendimiento de entregas y comportamiento de pagos

---

## Setup

In [1]:
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

# Cambia 'tu_usuario' por tu usuario de Mac
%sql postgresql://tomas@localhost:5432/data_engineering

---
## 1. Overview del dataset

In [2]:
%%sql
-- Volumen general por tabla
SELECT 'orders'    AS tabla, COUNT(*) AS filas FROM orders
UNION ALL
SELECT 'customers' AS tabla, COUNT(*) AS filas FROM customers
UNION ALL
SELECT 'payments'  AS tabla, COUNT(*) AS filas FROM payments;

,tabla,filas
0,payments,103886
1,customers,99441
2,orders,96478


In [ ]:
%%sql
-- Ver columnas y tipos de la tabla orders
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_name = 'orders'
ORDER BY ordinal_position;*

In [3]:
%%sql
-- Rango temporal del dataset
SELECT
    MIN(order_purchase_timestamp) AS primer_pedido,
    MAX(order_purchase_timestamp) AS ultimo_pedido
FROM orders;

,primer_pedido,ultimo_pedido
0,2016-09-15 12:16:38,2018-08-29 15:00:37


---
## 2. Análisis de entregas

In [4]:
%%sql
-- Distribución de tiempos de entrega
SELECT
    COUNT(*)                                        AS total_pedidos,
    ROUND(AVG(delivery_days)::numeric, 1)           AS media_dias,
    PERCENTILE_CONT(0.5) WITHIN GROUP
        (ORDER BY delivery_days)                    AS mediana_dias,
    MIN(delivery_days)                              AS min_dias,
    MAX(delivery_days)                              AS max_dias
FROM orders;

,total_pedidos,media_dias,mediana_dias,min_dias,max_dias
0,96478,12.1,10.0,0.0,209.0


In [5]:
%%sql
-- Pedidos agrupados por rango de días de entrega
SELECT
    CASE
        WHEN delivery_days = 0        THEN '0 días (mismo día)'
        WHEN delivery_days BETWEEN 1 AND 7   THEN '1-7 días'
        WHEN delivery_days BETWEEN 8 AND 14  THEN '8-14 días'
        WHEN delivery_days BETWEEN 15 AND 30 THEN '15-30 días'
        ELSE '+ 30 días'
    END AS rango_entrega,
    COUNT(*) AS total,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS porcentaje
FROM orders
GROUP BY rango_entrega
ORDER BY MIN(delivery_days);

,rango_entrega,total,porcentaje
0,0 días (mismo día),13,0.0
1,1-7 días,33683,34.9
2,8-14 días,36397,37.7
3,15-30 días,22260,23.1
4,+ 30 días,4125,4.3


In [6]:
%%sql
-- Top 10 ciudades con más pedidos y su media de entrega
-- JOIN entre orders y customers por customer_id
SELECT
    c.customer_city                                 AS ciudad,
    COUNT(*)                                        AS total_pedidos,
    ROUND(AVG(o.delivery_days)::numeric, 1)         AS media_dias_entrega
FROM orders o
LEFT JOIN customers c ON o.customer_id = c.customer_id
GROUP BY c.customer_city
ORDER BY total_pedidos DESC
LIMIT 10;

,ciudad,total_pedidos,media_dias_entrega
0,sao paulo,15045,7.6
1,rio de janeiro,6601,14.3
2,belo horizonte,2697,10.7
3,brasilia,2071,12.5
4,curitiba,1489,10.0
5,campinas,1406,9.4
6,porto alegre,1342,15.5
7,salvador,1188,18.9
8,guarulhos,1144,7.5
9,sao bernardo do campo,911,7.6


In [7]:
%%sql
-- Estados con peor tiempo medio de entrega (mínimo 100 pedidos)
SELECT
    c.customer_state                                AS estado,
    COUNT(*)                                        AS total_pedidos,
    ROUND(AVG(o.delivery_days)::numeric, 1)         AS media_dias_entrega
FROM orders o
LEFT JOIN customers c ON o.customer_id = c.customer_id
GROUP BY c.customer_state
HAVING COUNT(*) >= 100
ORDER BY media_dias_entrega DESC
LIMIT 10;

,estado,total_pedidos,media_dias_entrega
0,AM,145,26.0
1,AL,397,24.0
2,PA,946,23.3
3,MA,717,21.1
4,SE,335,21.0
5,CE,1279,20.8
6,PB,517,20.0
7,PI,476,19.0
8,RO,243,18.9
9,BA,3256,18.9


---
## 3. Análisis de pagos

In [8]:
%%sql
-- Distribución por método de pago
SELECT
    payment_type                                    AS metodo_pago,
    COUNT(*)                                        AS total_transacciones,
    ROUND(AVG(payment_value)::numeric, 2)           AS valor_medio,
    ROUND(SUM(payment_value)::numeric, 2)           AS valor_total
FROM payments
GROUP BY payment_type
ORDER BY total_transacciones DESC;

,metodo_pago,total_transacciones,valor_medio,valor_total
0,credit_card,76795,163.32,12542084.19
1,boleto,19784,145.03,2869361.27
2,voucher,5775,65.70,379436.87
3,debit_card,1529,142.57,217989.79
4,not_defined,3,0.00,0.00


In [9]:
%%sql
-- Pedidos con múltiples métodos de pago
SELECT
    COUNT(*) AS total_pagos,
    payment_installments AS cuotas,
    ROUND(AVG(payment_value)::numeric, 2) AS valor_medio
FROM payments
GROUP BY payment_installments
ORDER BY cuotas;

,total_pagos,cuotas,valor_medio
0,2,0,94.32
1,52546,1,112.42
2,12413,2,127.23
3,10461,3,142.54
4,7098,4,163.98
5,5239,5,183.47
6,3920,6,209.85
7,1626,7,187.67
8,4268,8,307.74
9,644,9,203.44


In [11]:
%%sql
-- Pedidos agrupados por número de cuotas
SELECT
    payment_installments AS cuotas,
    COUNT(*) AS total_pagos,
    ROUND(AVG(payment_value)::numeric, 2) AS valor_medio
FROM payments
GROUP BY payment_installments
ORDER BY cuotas;

,cuotas,total_pagos,valor_medio
0,0,2,94.32
1,1,52546,112.42
2,2,12413,127.23
3,3,10461,142.54
4,4,7098,163.98
5,5,5239,183.47
6,6,3920,209.85
7,7,1626,187.67
8,8,4268,307.74
9,9,644,203.44


---
## 4. JOINs combinados — Orders + Customers + Payments

In [10]:
%%sql
-- Valor medio de pedido por estado, con tiempo de entrega
-- Triple JOIN: orders → customers → payments
SELECT
    c.customer_state                                AS estado,
    COUNT(DISTINCT o.order_id)                      AS total_pedidos,
    ROUND(AVG(o.delivery_days)::numeric, 1)         AS media_dias_entrega,
    ROUND(AVG(p.payment_value)::numeric, 2)         AS valor_medio_pedido
FROM orders o
LEFT JOIN customers c ON o.customer_id = c.customer_id
LEFT JOIN payments p  ON o.order_id   = p.order_id
GROUP BY c.customer_state
ORDER BY total_pedidos DESC;

,estado,total_pedidos,media_dias_entrega,valor_medio_pedido
0,SP,40501,8.3,136.39
1,RJ,12350,14.9,158.08
2,MG,11354,11.5,154.12
3,RS,5345,14.8,155.45
4,PR,4923,11.6,152.45
5,SC,3546,14.5,162.58
6,BA,3256,18.8,169.76
7,DF,2080,12.5,161.60
8,ES,1995,15.4,153.62
9,GO,1957,15.1,163.31


---
## 5. Windows Function

> Añade aquí tus propias queries de análisis.

In [2]:
%%sql
-- Ranking de pedidos por tiempo de entrega dentro de cada ciudad
-- ROW_NUMBER asigna 1 al más rápido, 2 al segundo más rápido, etc.
SELECT
    order_id,
    customer_id,
    delivery_days,
    ROW_NUMBER() OVER (ORDER BY delivery_days ASC) AS ranking_global
FROM orders
LIMIT 20;


The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


,order_id,customer_id,delivery_days,ranking_global
0,d3ca7b82c922817b06e5ca21165c5ea2,d23df2c6c3e51d875f458d123b2b3c90,0.0,1
1,38c1e3d4ed6a13cd0cf612d4c09766e9,18c934f4cdc994cd04eb13bce3f47a18,0.0,2
2,b70a8d75313560b4acf607739560a0e5,42992f7eb57b0f04f5a52cf89112f415,0.0,3
3,8339b608be0d84fca9d8da68b58332c3,ff58662c328f81d3ee549c9caa942f39,0.0,4
4,e65f1eeee1f52024ad1dcd03447f7482,198f511b5a75bf936a96f1d4769e3974,0.0,5
5,1d893dd7ca5f77ebf5f59f0d2017eee0,b19da0df0271e8a3553e3670f86aeab5,0.0,6
6,79e324907160caea526fd8b94389dbbc,331d79b67223ee7e5cd31d3e03e4cfcc,0.0,7
7,f3c6775ba3d2d9fe2826f93b71f12008,6aef84c09844a371d82a49152c550b95,0.0,8
8,434cecee7d1a65fc65358a632b6f725f,922a46283625e9c096bfd998913c470c,0.0,9
9,f349cdb62f69c3fae5c4d7d3f3a4a185,c5e200d485ae35a7036cc2e7c1d8ea81,0.0,10


In [4]:
%%sql
-- Ranking de pedidos por tiempo de entrega DENTRO de cada ciudad
-- PARTITION BY reinicia el contador en cada ciudad
SELECT
    o.order_id,
    c.customer_city,
    o.delivery_days,
    ROW_NUMBER() OVER (
        PARTITION BY c.customer_city
        ORDER BY o.delivery_days ASC
    ) AS ranking_en_ciudad
FROM orders o
LEFT JOIN customers c ON o.customer_id = c.customer_id
LIMIT 30;


,order_id,customer_city,delivery_days,ranking_en_ciudad
0,d99e6849f7676dade195f20c26f0eb4f,abadia dos dourados,5.0,1
1,0a9a43ac5fe59c6c4bee2a8f9b9fcce8,abadia dos dourados,8.0,2
2,50ba38c4dc467baab1ea2c8c7747934d,abadia dos dourados,21.0,3
3,3f1294f87d79b57f5d55ba7b80c3d94f,abadiania,29.0,1
4,6c12feac9a308e1382d9b19cca7f20b2,abaete,4.0,1
5,4ccc4e96e0fa5c35efbf9bf836ebef72,abaete,7.0,2
6,807756ebd577f025334944b87da7dbc3,abaete,7.0,3
7,1c2f555edfc445e72867c301b68ac512,abaete,7.0,4
8,74bdc516d1c6a8e16cd1a546ea067c26,abaete,7.0,5
9,5525ea8ee9e6150faa9b41dfc4024552,abaete,8.0,6


In [6]:
%%sql
-- Dame el pedido más rápido de cada ciudad
WITH pedidos_ranked AS (
    -- Aquí va una query COMPLETA
    SELECT
        o.order_id,
        c.customer_city,
        o.delivery_days,
        ROW_NUMBER() OVER (
            PARTITION BY c.customer_city
            ORDER BY o.delivery_days ASC
        ) AS ranking_en_ciudad
    FROM orders o
    LEFT JOIN customers c ON o.customer_id = c.customer_id
)
SELECT *
FROM pedidos_ranked
WHERE ranking_en_ciudad = 1;

,order_id,customer_city,delivery_days,ranking_en_ciudad
0,d99e6849f7676dade195f20c26f0eb4f,abadia dos dourados,5.0,1
1,3f1294f87d79b57f5d55ba7b80c3d94f,abadiania,29.0,1
2,6c12feac9a308e1382d9b19cca7f20b2,abaete,4.0,1
3,57d0f29d2d770802ab8e393778fc9052,abaetetuba,10.0,1
4,73ffc468078b5f3e2c4260065e5e5152,abaiara,25.0,1
...,...,...,...,...
4080,26968873aef9380ae17bfc3717c8cd7d,xinguara,12.0,1
4081,9053095d6a8816107d406e70df00efd0,xique-xique,15.0,1
4082,388a9117213936e0290e6e7d50f9fee0,zacarias,9.0,1
4083,fa1a82991080d1ed588c811aa7e8fdb6,ze doca,14.0,1


In [10]:
%%sql
-- Los 3 pedidos más lentos de cada estado
WITH pedidos_ranked AS (
    -- Aquí va una query COMPLETA
    SELECT
        o.order_id,
        c.customer_state,
        o.delivery_days,
        ROW_NUMBER() OVER (
            PARTITION BY c.customer_state
            ORDER BY o.delivery_days DESC NULLS LAST
        ) AS ranking_en_estado
    FROM orders o
    LEFT JOIN customers c ON o.customer_id = c.customer_id
)
SELECT *
FROM pedidos_ranked
WHERE ranking_en_estado <= 3;

,order_id,customer_state,delivery_days,ranking_en_estado
0,0b7cb1c7ea62d95ba39702f4ae850108,AC,72.0,1
1,2bcf1f79964f8b4f4f62f19441601b1a,AC,66.0,2
2,ca2c921a5b9d60eba417d106c724c6f6,AC,42.0,3
3,d573a47c29466bb2e4e46aef0ff04f23,AL,90.0,1
4,ae213a9f84c777fb7a31e8c0f09fd30c,AL,77.0,2
...,...,...,...,...
76,2fe324febf907e3ea3f2aa9650869fa5,SP,189.0,2
77,d24e8541128cea179a11a65176e0a96f,SP,175.0,3
78,f5691c2b1ca263490374d13d020bd950,TO,58.0,1
79,1734eb8be40c91938e56f9d245b636a5,TO,44.0,2


In [11]:
%%sql
-- Evolución mensual de pedidos con comparación al mes anterior
WITH pedidos_por_mes AS (
    SELECT
        DATE_TRUNC('month', order_purchase_timestamp) AS mes,
        COUNT(*) AS total_pedidos
    FROM orders
    GROUP BY mes
    ORDER BY mes
)
SELECT
    mes,
    total_pedidos,
    LAG(total_pedidos) OVER (ORDER BY mes) AS pedidos_mes_anterior,
    total_pedidos - LAG(total_pedidos) OVER (ORDER BY mes) AS diferencia
FROM pedidos_por_mes;

,mes,total_pedidos,pedidos_mes_anterior,diferencia
0,2016-09-01,1,NaN,NaN
1,2016-10-01,265,1.0,264.0
2,2016-12-01,1,265.0,-264.0
3,2017-01-01,750,1.0,749.0
4,2017-02-01,1653,750.0,903.0
5,2017-03-01,2546,1653.0,893.0
6,2017-04-01,2303,2546.0,-243.0
7,2017-05-01,3546,2303.0,1243.0
8,2017-06-01,3135,3546.0,-411.0
9,2017-07-01,3872,3135.0,737.0


In [12]:
%%sql
-- Diferencia entre ROW_NUMBER, RANK y DENSE_RANK
-- Usamos delivery_days para ver cómo tratan los empates
SELECT
    order_id,
    delivery_days,
    ROW_NUMBER()  OVER (ORDER BY delivery_days ASC NULLS LAST) AS row_number,
    RANK()        OVER (ORDER BY delivery_days ASC NULLS LAST) AS rank,
    DENSE_RANK()  OVER (ORDER BY delivery_days ASC NULLS LAST) AS dense_rank
FROM orders
LIMIT 20;

,order_id,delivery_days,row_number,rank,dense_rank
0,f349cdb62f69c3fae5c4d7d3f3a4a185,0.0,1,1,1
1,d3ca7b82c922817b06e5ca21165c5ea2,0.0,2,1,1
2,38c1e3d4ed6a13cd0cf612d4c09766e9,0.0,3,1,1
3,e65f1eeee1f52024ad1dcd03447f7482,0.0,4,1,1
4,434cecee7d1a65fc65358a632b6f725f,0.0,5,1,1
5,f3c6775ba3d2d9fe2826f93b71f12008,0.0,6,1,1
6,bb5a519e352b45b714192a02ffe25681,0.0,7,1,1
7,b70a8d75313560b4acf607739560a0e5,0.0,8,1,1
8,21a8ffca665bc7a1087d31751a7b7cbc,0.0,9,1,1
9,1d893dd7ca5f77ebf5f59f0d2017eee0,0.0,10,1,1


In [ ]:
WITH pedidos_ranked AS (
    -- Aquí va una query COMPLETA
    SELECT
        o.order_id,
        c.customer_state,
        o.delivery_days,
        ROW_NUMBER() OVER (
            PARTITION BY c.customer_state
            ORDER BY o.delivery_days DESC NULLS LAST
        ) AS ranking_en_estado
    FROM orders o
    LEFT JOIN customers c ON o.customer_id = c.customer_id
)
SELECT *
FROM pedidos_ranked
WHERE ranking_en_estado <= 3;

In [39]:
%%sql
-- Top 3 ciudades con más pedidos por estado, mostrando el total de pedidos de cada ciudad
WITH ciudades_contadas AS (
    -- Paso 1: contar pedidos por ciudad
    SELECT
        c.customer_state,
        c.customer_city,
        COUNT(*) AS total_pedidos
    FROM orders o
    LEFT JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY c.customer_state, c.customer_city
),
ciudades_ranked AS (
    -- Paso 2: añadir el ranking
    SELECT
        customer_state,
        customer_city,
        total_pedidos,
        DENSE_RANK() OVER (
            PARTITION BY customer_state
            ORDER BY total_pedidos DESC
        ) AS ranking
    FROM ciudades_contadas
)
-- Paso 3: filtrar top 3
SELECT *
FROM ciudades_ranked
WHERE ranking <= 3
ORDER BY customer_state, ranking;

,customer_state,customer_city,total_pedidos,ranking
0,AC,rio branco,69,1
1,AC,cruzeiro do sul,3,2
2,AC,senador guiomard,2,3
3,AC,xapuri,2,3
4,AL,maceio,236,1
...,...,...,...,...
77,SP,campinas,1406,2
78,SP,guarulhos,1144,3
79,TO,palmas,88,1
80,TO,araguaina,38,2


In [28]:
%%sql
SELECT
    c.customer_state,
    c.customer_city,
    COUNT(*) AS total_pedidos
FROM orders o
LEFT JOIN customers c ON o.customer_id = c.customer_id
GROUP BY c.customer_state, c.customer_city
ORDER BY c.customer_state, total_pedidos DESC
LIMIT 20;

,customer_state,customer_city,total_pedidos
0,AC,rio branco,69
1,AC,cruzeiro do sul,3
2,AC,senador guiomard,2
3,AC,xapuri,2
4,AC,epitaciolandia,1
5,AC,manoel urbano,1
6,AC,brasileia,1
7,AC,porto acre,1
8,AL,maceio,236
9,AL,arapiraca,29


In [41]:
%%sql
-- Ver columnas y tipos de la tabla orders
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_name = 'orders'
ORDER BY ordinal_position;

,column_name,data_type
0,order_id,text
1,customer_id,text
2,order_status,text
3,order_purchase_timestamp,timestamp without time zone
4,order_approved_at,timestamp without time zone
5,order_delivered_carrier_date,timestamp without time zone
6,order_delivered_customer_date,timestamp without time zone
7,order_estimated_delivery_date,timestamp without time zone
8,delivery_days,double precision


In [51]:
%%sql
SELECT
CASE
    WHEN delivery_days < 7 THEN 'rapido'
    WHEN delivery_days BETWEEN 7 AND 20 THEN 'normal'
    WHEN delivery_days > 20 THEN 'lento'
    END AS rango_entrega,
    count(*) as total_pedidos
from orders
GROUP BY rango_entrega
order by total_pedidos DESC

,rango_entrega,total_pedidos
0,normal,57926
1,rapido,26045
2,lento,12499
3,None,8


In [67]:
%%sql
select order_id, delivery_days
    from orders
    where delivery_days >
(
    select AVG(delivery_days)
from orders
)

,order_id,delivery_days
0,53cdb2fc8bc7dce0b6741e2150273451,13.0
1,949d5b44dbf5de918fe9c16f97b45f8a,13.0
2,a4591c265e18cb1dcee52889e2d8acc3,16.0
3,e69bfb5eb88e0ed6a785585b27e16dbf,18.0
4,dcb36b511fcac050b97cd5c05de84dc3,13.0
...,...,...
34583,cfa78b997e329a5295b4ee6972c02979,37.0
34584,9115830be804184b91f5c00f6f49f92d,16.0
34585,63943bddc261676b46f01ca7ac2f7bd8,22.0
34586,83c1379a015df1e13d02aae0204711ab,24.0


In [66]:
%%sql
--Pedidos que tardaron más que la media global
select AVG(delivery_days)
from orders

,avg
0,12.093604
